# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa — Exploration with `mlcroissant`

This notebook provides a reproducible pipeline for loading and exploring a Croissant-based dataset using the `mlcroissant` library. All field and entity references use their `@id` fields for clarity, ensuring robust and unambiguous access within the dataset schema.

### Dataset Source
The dataset source is provided via the Croissant schema URL below:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records with `mlcroissant`. For seamless reproducibility, all data and schema access is via the authoritative Croissant URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset from schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Pretty-print dataset name and description
print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}")
print(f"\nPublished: {metadata.datePublished}\nLicense: {metadata.license}")

## 2. Data Overview
The Croissant schema organizes records into record sets. We'll enumerate available record sets using their `@id`, along with their columns and fields. For all data access, we reference entities by their `@id`.

**Note:** Record sets correspond to tabular collections (e.g., `cr:RecordSet`).

In [ ]:
record_sets = list(dataset.metadata.recordSets)

print(f"Available record sets (by @id): {len(record_sets)}\n")
for rs in record_sets:
    print(f"  RecordSet @id: {rs.id}")
    print(f"    Name: {getattr(rs, 'name', 'N/A')}")
    print(f"    Description: {getattr(rs, 'description', 'N/A')}")
    print(f"    Columns/Fields by @id:")
    for fld in rs.fields:
        print(f"      - {fld.id} ({getattr(fld, 'name', 'N/A')})")
    print()

### Sample records (all fields by `@id`)
Here we print a preview of the records using the main record set's `@id` (see cell above for available record sets).

This helps users identify which keys to use for further data extraction.

In [ ]:
# Set the main record set @id (replace as needed based on the overview above)
# We'll select the first record set for demonstration
main_recordset_id = record_sets[0].id if record_sets else None

if main_recordset_id:
    print(f"Sample records for record set @id: {main_recordset_id}")
    sample = []
    for i, rec in enumerate(dataset.records(record_set=main_recordset_id)):
        sample.append(rec)
        if i >= 2:
            break
    pp = pprint.PrettyPrinter()
    pp.pprint(sample)
else:
    print("ERROR: No record sets found in the dataset.")

## 3. Data Extraction
Here we load each tabular record set into a `pandas.DataFrame`. Each record set is referenced by its `@id`.

We preview the column headers of each loaded DataFrame — all field names are the Croissant `@id` values.

In [ ]:
# Load data from all record sets into DataFrames
dataframes = {}

for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded DataFrame for RecordSet @id: {rs.id}")
    print(f"  Shape: {df.shape}")
    print(f"  Columns (@id): {list(df.columns)}\n")

Let's preview the head of the main record set (first detected) for quick inspection.

In [ ]:
if main_recordset_id:
    display(dataframes[main_recordset_id].head())
else:
    print("No main record set to display.")

## 4. Exploratory Data Analysis (EDA)

Let's perform some basic data processing and feature engineering. All variable references use Croissant field `@id` strings.

- We'll select an example numeric field (e.g., one of the psychometric scores such as `gad_7_score`, `phq_9_score` or similar; reference @ids from earlier output).
- We filter by a threshold, normalize, and group by a categorical variable (e.g., `gender`).


In [ ]:
# Example: Suppose one field @id is 'gad_7_score', and a group field is 'gender'.
# You should update these variable values to match actual @ids from your dataset above.

example_numeric_field = None
example_group_field = None

if main_recordset_id is not None:
    df = dataframes[main_recordset_id]
    # Try and guess a likely numeric field, e.g., one that looks like a score
    for col in df.columns:
        col_lc = col.lower()
        if 'gad' in col_lc or 'phq' in col_lc or 'psq' in col_lc or 'score' in col_lc:
            example_numeric_field = col
            break

    # And a group-by field (e.g. gender)
    for col in df.columns:
        col_lc = col.lower()
        if 'gender' in col_lc or 'sex' in col_lc:
            example_group_field = col
            break
    
    print(f"Using numeric field @id: {example_numeric_field}")
    print(f"Grouping by field @id: {example_group_field}\n")
    
    # Proceed with analysis if the field exists and is numeric
    if example_numeric_field and pd.api.types.is_numeric_dtype(df[example_numeric_field]):
        threshold = df[example_numeric_field].median()  # Use median as a threshold for illustration
        filtered_df = df[df[example_numeric_field] > threshold].copy()
        print(f"Filtered records where {example_numeric_field} > {threshold}")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{example_numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[example_numeric_field] - filtered_df[example_numeric_field].mean()) / filtered_df[example_numeric_field].std()
        print(f"Normalized '{example_numeric_field}':")
        display(filtered_df[[example_numeric_field, norm_col]].head())

        # Group
        if example_group_field and example_group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(example_group_field).mean(numeric_only=True)
            print(f"Mean statistics for records grouped by {example_group_field}:\n")
            display(grouped_df.head())
        else:
            print("No appropriate group field detected for grouping.")
    else:
        print(f"Field '{example_numeric_field}' not found or not numeric. Please review the DataFrame columns to set the correct field.")
else:
    print("No main record set for EDA.")

## 5. Visualization

Visualize the distribution of the selected numeric field and, if appropriate, the field by groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if main_recordset_id and example_numeric_field:
    df = dataframes[main_recordset_id]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[example_numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of '{example_numeric_field}'")
    plt.xlabel(example_numeric_field)
    plt.show()

    if example_group_field and example_group_field in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[example_group_field], y=df[example_numeric_field])
        plt.title(f"'{example_numeric_field}' by '{example_group_field}'")
        plt.xlabel(example_group_field)
        plt.ylabel(example_numeric_field)
        plt.show()
else:
    print("Unable to plot numeric field. Please check previous steps for correct field selection.")

## 6. Conclusion

- We loaded this mental health survey dataset using the Croissant schema and `mlcroissant`.
- Key tabular data was explored by referencing all fields via their `@id`, ensuring compatibility and schema stability.
- Basic EDA was performed, including filtering, normalization, grouping, and visual exploration.

For more advanced analysis, consult the Croissant documentation for formal entity references and consider exploring additional record sets and relationships.